In [1]:
%%HTML
### Cambiamos los tamaños de fuente asociados a las tablas
<style>
    tr { font-size: 18px; background-color: lightblue; font-color: orange}
    td { font-size: 18px; }
</style>

<div class="alert alert-block alert-info">
<b>✅ NOTA</b>: Este notebook se ha validado con un entorno creado en Anaconda con Python 3.12.X.
</div>

# Biblioteca de Mapas

Incluye este fichero, ejecutándolo con `%run RCR_Bib_Mapas.ipynb`, para tener disponibles las funciones de utilidad que vimos en el **Boletín 1**.

## [Folium](http://python-visualization.github.io/folium/)
Biblioteca que permite visualizar datos en un mapa interactivo.

In [2]:
try:
    import folium
except:
    !pip install folium==0.20.0
    import folium

### Creación de Mapa:

* Argumentos: 
    * `center`: coordenadas del centro del mapa.
    * `zoom`: nivel de zoom del mapa.
* Valor de retorno: mapa

In [3]:
def create_map(center, zoom=14):
    m = folium.Map(location=list(reversed(center)), zoom_start=zoom)
    return m

### Añadir marcador en el mapa:
* Argumentos:
    * `m`: mapa que habrá sido creado anteriormente.
    * `coords`: coordenadas en las que se quiere poner el marcador.
    * `text`: texto con el que queremos etiquetar la marca en el mapa.
    * `color`: color del marcador.
    * `icon`: icono que se dibujará para indicar el marcador.
* Valor de retorno: mapa actualizado con el marcador.

In [4]:
def add_marker(m, coords, text="", color="red", icon="info-sign"):
    folium.map.Marker(list(reversed(coords)),
                      icon=folium.Icon(color=color,
                                       icon=icon
                                       ),
                      popup=text,
                      ).add_to(m)
    return m

Por defecto se usan los iconos **Glyphicon** de [Bootstrap](https://getbootstrap.com/docs/3.3/components/).

También se pueden usas los iconos [**Font Awesome**](https://fontawesome.com/search?ic=free&o=r) (obviamente, tan sólo los marcados como gratis).

In [5]:
def add_marker_fa(m, coords, text="", color="red", icon="info-sign"):
    folium.map.Marker(list(reversed(coords)),
                      icon=folium.Icon(color=color,
                                       icon=icon,
                                       prefix='fa'
                                       ),
                      popup=text,
                      ).add_to(m)
    return m

### Crear una línea entre dos puntos de interés del mapa:
* Argumentos:
    * `m`: mapa que habrá sido creado anteriormente.
    * `coord_origen`: coordenadas del punto origen de la línea.
    * `coord_destino`: coordenadas del punto destino de la línea.
    * `color`: color del marcador (rojo por defecto).
* Valor de retorno: mapa actualizado con el marcador.

In [6]:
def add_line(m, coord_origen, coord_destino, color="red"):
    l = [list(reversed(coord_origen))] + [list(reversed(coord_destino))]
    folium.PolyLine(l, color=color, weight = 2.5, opacity = 1).add_to(m)
    return m

### Crear una línea entre los puntos de interés de una ruta del mapa:
* Argumentos:
    * `m`: mapa que habrá sido creado anteriormente.
    * `ruta`: lista con las coordenadas de los puntos de interés de la ruta.
    * `color`: color de la línea (rojo por defecto).
* Valor de retorno: mapa actualizado con la línea.

In [7]:
def add_ruta(m, ruta, color="red"):
    rev_ruta = []
    for x in ruta:
        rev_ruta.append(list(reversed(x)))
        
    folium.PolyLine(rev_ruta, color=color).add_to(m)
    return m    

### save_map

Permite guardar en un fichero HTML un mapa. 


* Argumentos:
    * m: mapa a guardar a fichero.
    * file: fichero HTML donde se guardará el mapa.

In [8]:
def save_map(m, file):
    m.save(file)
    #print("Mapa creado")

## Openrouteservice

Biblioteca que incluye funciones para calcular posiciones en un mapa a partir de una dirección, distancias, ...

In [9]:
try:
    import openrouteservice
except:
    !pip install openrouteservice==2.3.3
    import openrouteservice

### Geocoding

Proceso de convertir una dirección en coordenadas geográficas (longitud, latitud).

* Argumentos:

    * `client`: instancia del cliente, resultado de invocar a la función `openrouteservice.Client()` con el argumento `key` indicando la clave del cliente correspondiente. 
    * `place`: dirección del lugar del que queremos sus coordenadas geográficas.
    * `focus`: coordenadas geográficas del punto alrededor del cual se va a focalizar la búsqueda de la dirección indicada en `place`, con un radio de 100 km.

* Valor de retorno:
    Coordenadas (longitud, latitud) del punto especificado en `place`.

<div class="alert alert-block alert-danger">
<b>NOTA</b>: crear una cuenta en <A href="http://openrouteservice.org"> openrouteservice </A> y entrar en ella. Después crear un <tt>token</tt> y copiar la clave que genere asociada a él y usarla como valor para el argumento <tt>key</tt> en la invocación a la función <tt>openrouteservice.Client()</tt>.
</div>

In [10]:
def geocoding(client, place, focus):
    try:
        #r = openrouteservice.geocode.pelias_structured(client,
        #                    address=place['calle'], locality=place['ciudad'],
        #                    postalcode=place['cp'], country=place['pais'])
        r = openrouteservice.geocode.pelias_search(client, text=place, 
                                                   focus_point=focus)
        coords = r['features'][0]['geometry']['coordinates']
        return coords
    except:
        return None

### AntPath

Función de la biblioteca `folium.plugins` que permite añadir un camino al mapa que venga marcado por líneas discontínuas animadas.

* Argumentos:
    * locations: lista de las coordenadas geográficas de los puntos a incluir en el camino.
* Valor de retorno:
    El mapa actualizado.

In [11]:
import folium.plugins

### Crear un camino de hormigas entre los puntos de interés de una ruta del mapa:
* Argumentos:
    * `m`: mapa que habrá sido creado anteriormente.
    * `ruta`: lista con las coordenadas de los puntos de interés de la ruta.
    * `color`: color del camino de hormigas (rojo por defecto).
* Valor de retorno: mapa actualizado con el camino de hormigas.

In [12]:
def add_ruta_ant(m, ruta, color="red"):
    rev_ruta = []
    for x in ruta:
        rev_ruta.append(list(reversed(x)))
        
    folium.plugins.AntPath(rev_ruta, color=color).add_to(m)
    return m   

### distancia_entre

Esta función permite calcular la distancia en metros entre un punto origen y otro destino.

* Argumentos:
    * or_long: longitud del punto origen.
    * or_lat: latitud del punto origen.
    * dst_long: longitud del punto destino.
    * dst_lat: latitud del punto destino.

* Valor de retorno:
    Distancia en metros entre los dos puntos indicados como argumento.

In [13]:
def distancia_entre ( client, or_long, or_lat, dst_long, dst_lat):
    locat_rev = [[or_lat, or_long]]
    locat_rev.append([dst_lat, dst_long])
    
    dm = openrouteservice.distance_matrix.distance_matrix(client, locat_rev, 'foot-walking', [0], [1],
                                                      metrics=['distance', 'duration'])
    
    return dm['distances'][0][0]

### distancia

Esta función permite calcular la distancia en metros entre un punto origen y otro destino.

* Argumentos:
    * lat1: latitud del punto origen.
    * lon1: longitud del punto origen.
    * lat2: latitud del punto destino.
    * long2: longitud del punto destino.

* Valor de retorno:
    Distancia en metros entre los dos puntos indicados como argumento.

In [14]:
try:
    import geopy
except:
    !pip install geopy==2.4.1
    import geopy

In [15]:
import geopy.distance
def distancia(lon1, lat1, lon2, lat2):
    return geopy.distance.geodesic((float(lat1), float(lon1)),
                                    (float(lat2), float(lon2))).meters

### Crear una ruta que se ajuste al recorrido real en el mapa y pintarla en el mapa como un camino de hormigas:
* Argumentos:
    * `mapa`: mapa que habrá sido creado anteriormente.
    * `client`: instancia del cliente, resultado de invocar a la función `openrouteservice.Client()` con el argumento `key` indicando la clave del cliente correspondiente. 
    * `coordenadas`: lista de coordenadas de los puntos de interés que queremos unir por la ruta.
    * `color`: color del camino de hormigas (rojo por defecto).
    * `tipo`: tipo de vehículo a usar para el recorrido. Posibles valores: “driving-car”, “driving-hgv”, “foot-walking”, “foot-hiking”, “cycling-regular”, “cycling-road”,”cycling-mountain”, “cycling-electric”. Valor por defecto: "foot-walking".

* Valores de retorno:
    * Diccionario con los siguientes campos:
        * `distancia`: distancia total de la ruta en metros.
        * `mapa`: mapa actualizado con el camino de hormigas de la ruta calculada.
        * `ruta`: lista con las coordenadas de los puntos a recorrer para la ruta.

In [16]:
def crea_ruta (mapa, client, coordenadas, color='red', tipo='foot-walking'):
    # tipo es un valor entre 
    ### [“driving-car”, “driving-hgv”, “foot-walking”, “foot-hiking”, “cycling-regular”, 
    ### “cycling-road”,”cycling-mountain”, “cycling-electric”]

    route = client.directions(coordinates=coordenadas,
                                  profile=tipo,
                                  format='geojson')
    
    ruta = route['features'][0]['geometry']['coordinates']
    mapa = add_ruta_ant(mapa, ruta, color=color)
    
    suma = 0
    segments = route['features'][0]['properties']['segments']
    for i in segments:
        print(i['distance'])
        suma = suma + i['distance']

    return {'distancia': suma, 'mapa': mapa, 'ruta': ruta}